## Share of New Cars Sold that are Electric

In this notebook, we will explore the Share of New Cars Sold That Are Electric dataset from Our World in Data. The dataset tracks the percentage of new car sales that are electric for each country and year. Here, electric cars include fully battery-electric and plug-in hybrids. Our World in Data gets this data from the International Energy Agency, which publishes its Global EV Outlook every year.

This dataset matters because switching from gas to electric cars is one of the main ways countries are trying to cut emissions from transportation. Adoption also varies a lot around the world. In Norway, almost every new car is an electric one. In China, more than half are, while in the United States, it’s just 10%. Worldwide, one in four (25%) cars sold in 2025 were electric, more than double the share from just four years earlier.

In this notebook, we will load the data, describe its structure, and do some preliminary cleaning. Then we will look at patterns over time and across countries. Later, this dataset will be combined with our group’s data on transport CO₂ emissions and fossil fuel consumption, to see whether higher EV adoption lines up with lower emissions and fuel use.

In [1]:
import pandas as pd
import numpy as np 

### EDA

In [2]:
# Load the dataset 
evs = pd.read_csv("electric-car-sales-share.csv")

evs.head()

,Entity,Code,Year,Share of new cars that are electric
0,Australia,AUS,2011,0.0061
1,Australia,AUS,2012,0.0290
2,Australia,AUS,2013,0.0330
3,Australia,AUS,2014,0.1500
4,Australia,AUS,2015,0.1900


In [3]:
evs.isna().sum()

Entity                                  0
Code                                   12
Year                                    0
Share of new cars that are electric     0
dtype: int64

 #### Data description 
The dataset has 817 rows and 4 columns. Each row is one country or region in one year, for example Australia in 2011.

- Entity: Country or region name
- Code: Three letter country code, like AUS
- Year: 2010 to 2025
- Share of new cars that are electric: Percentage of new cars sold that year that were battery electric or plug in hybrid

Source: International Energy Agency, Global EV Outlook, processed by Our World in Data. [Share of new cars sold that are electric](https://ourworldindata.org/grapher/electric-car-sales-share) 

### Data cleaning

In [4]:
# See which entities are missing a code
evs[evs["Code"].isna()]["Entity"].unique()

array(['Rest of World'], dtype=object)

In [5]:
# Check for region codes that start with OWID_
evs[evs["Code"].str.startswith("OWID_", na=False)]["Entity"].unique()

array(['Europe', 'European Union (27)', 'World'], dtype=object)

In [6]:
# Save World separately for comparison later
world = evs[evs["Entity"] == "World"]

# Drop Rest of World and any OWID region codes so only countries remain
evs_countries = evs.dropna(subset=["Code"])
evs_countries = evs_countries[~evs_countries["Code"].str.startswith("OWID_")]

# Rename the long column
evs_countries = evs_countries.rename(columns={"Share of new cars that are electric": "ev_share"})
world = world.rename(columns={"Share of new cars that are electric": "ev_share"})


In [7]:
# Number of years of data per country
evs_countries["Entity"].value_counts()

Entity
Brazil                  16
Belgium                 16
China                   16
Germany                 16
India                   16
Japan                   16
France                  16
Italy                   16
Portugal                16
New Zealand             16
Norway                  16
Poland                  16
Spain                   16
Sweden                  16
South Korea             16
Netherlands             16
United Kingdom          16
United States           16
Thailand                16
Switzerland             15
Denmark                 15
Israel                  15
Finland                 15
Mexico                  15
Colombia                15
Chile                   15
Canada                  15
Australia               15
Iceland                 14
United Arab Emirates    14
Malaysia                14
Turkey                  14
Russia                  13
South Africa            13
Greece                  13
Austria                 13
Costa Rica           

#### Select countries for project 

For this project, we will focus on 10 countries: Norway, China, the United States, Germany, the United Kingdom, Brazil, Thailand, Japan, India, and South Africa.

These were chosen to cover different regions and different levels of EV adoption. Norway leads the world in EV share. China is the largest EV market and manufacturer. The United States is included because it relies more on trucks and SUVs and has fewer affordable EV options. Germany and the United Kingdom represent large European markets, Brazil represents Latin America, and Thailand represents Southeast Asia. Japan and India are included because their adoption is still low. South Africa was chosen since it is one of the largest EV markets on the African continent and no data from the other countries was available

In [8]:
# Filter out for countries needed 
countries = [
    "Norway", "China", "United States", 
    "Germany", "South Africa", "United Kingdom", 
    "Brazil", "Thailand", "Japan", "India"]


evs_countries = evs_countries[evs_countries["Entity"].isin(countries)]

evs_countries["Entity"].unique()

array(['Brazil', 'China', 'Germany', 'India', 'Japan', 'Norway',
       'South Africa', 'Thailand', 'United Kingdom', 'United States'],
      dtype=object)

In [9]:
# Check the cleaned data
evs_countries.head()

,Entity,Code,Year,ev_share
44,Brazil,BRA,2010,0.00035
45,Brazil,BRA,2011,0.00028
46,Brazil,BRA,2012,0.00130
47,Brazil,BRA,2013,0.00240
48,Brazil,BRA,2014,0.00220


In [10]:
evs_countries.shape

(157, 4)

In [11]:
# Save the cleaned dataset 
evs_countries.to_csv("ev_share_clean.csv", index=False)

The only entity missing a code was Rest of World, which accounted for all 12 missing values. It groups together countries the IEA doesn’t report on individually, so it was removed. Europe, European Union (27), and World had OWID region codes, so they were also removed to keep one country per row. World was saved separately to use as a global benchmark. The long value column was renamed to ev_share to make it easier to work with.

After filtering to our 10 countries, the dataset has 157 rows and 4 columns. Nine countries have all 16 years from 2010 to 2025. South Africa has 13 years, so its early years are missing.

#### Summary 
Across our 10 countries, EV adoption ranges from almost all new cars in Norway to under a few percent in countries like India, Japan, and South Africa. China has grown very quickly in recent years, while the United States has grown more slowly. One pattern worth investigating is how fast each country moved from early adoption to higher shares, and why some countries took off while others stayed low.

This dataset alone only shows how many EVs are being sold, not whether they make a difference. It needs to be combined with the group’s data on transport CO₂ emissions per capita, fossil fuel consumption per capita, and the share of electricity from renewables. The datasets will be merged on Entity, Code, and Year. Since the transport data ends in 2023, the merged analysis will cover 2010 to 2023.

Goals for this dataset:

- Compare EV adoption trends across the 10 countries, using World as a benchmark
- Check whether countries with higher EV adoption also have cleaner electricity grids
- Check whether higher EV adoption lines up with lower transport emissions and fossil fuel use



